# Лабораторная 4 - Обучаемый фильтр
ФИО: ... | Вариант работы 3: ...

[Теория](THEORY.md) | [Задание](README.md) | [Презентация](lab4_theory_present.ipynb)

Код подготовки данных, модели, обучения и проверки приведен полностью. В разделе 1 задается собственный фильтр из работы 3. После выполнения заполняются описание фильтра и выводы по полученным результатам

Если PyTorch отсутствует: `%pip install torch` в отдельной ячейке

In [ ]:
from pathlib import Path
import numpy as np
import matplotlib.pyplot as plt
from scipy.signal import firwin, lfilter, freqz
plt.rcParams.update({'font.size': 13, 'figure.figsize': (10, 3.8),
                     'axes.grid': True, 'axes.unicode_minus': False})

import torch
from torch import nn
from torch.nn import functional as F
from copy import deepcopy
torch.manual_seed(42)
torch.set_num_threads(2)

## 1. Параметры фильтра
Ниже указан демонстрационный фильтр 80-160 Гц. Для своего варианта заменяются число коэффициентов, границы полосы и окно на выбранные в работе 3. Вместо вызова firwin можно перенести свой код расчета h

Описание выбранного фильтра и его назначения: ...

In [ ]:
fs = 1024.0  # Гц
numtaps = 65  # заменить параметрами своего фильтра
band = [80.0, 160.0]  # границы в Гц
window = ('kaiser', 8)

h = firwin(numtaps, band, pass_zero=False, fs=fs, window=window)


In [ ]:
h = np.asarray(h, dtype=np.float64)
if h.ndim != 1 or len(h) < 2 or not np.isfinite(h).all() or not np.any(h):
    raise ValueError('h должен быть ненулевым одномерным массивом конечных коэффициентов')
if not np.isfinite(fs) or fs <= 0:
    raise ValueError('Частота дискретизации должна быть положительной')
K = len(h)
length = max(512, 4*K)
print('Число коэффициентов:', K)
print('Частота дискретизации, Гц:', fs)
print('Длина обучающих записей:', length)

f, H = freqz(h, fs=fs, worN=2048)
fig, ax = plt.subplots(layout='constrained')
ax.plot(f, np.abs(H))
ax.set(xlabel='Частота, Гц', ylabel='Коэффициент передачи', title='АЧХ выбранного фильтра')
plt.show()


## 2. Данные
Входы - независимые случайные записи. Целевые выходы получаются применением выбранного фильтра. Формат тензоров: запись, канал, отсчет

In [ ]:
def make_pairs(seed,count,length,h):
    rng = np.random.default_rng(seed)
    x = rng.normal(0,0.3,(count,length)).astype(np.float32)
    y = lfilter(h,[1.0],x,axis=-1).astype(np.float32)
    return torch.from_numpy(x[:,None,:]),torch.from_numpy(y[:,None,:])

In [ ]:
train_x, train_y = make_pairs(110, 32, length, h)
val_x, val_y = make_pairs(120, 8, length, h)
print('Train:', tuple(train_x.shape), tuple(train_y.shape))
print('Validation:', tuple(val_x.shape), tuple(val_y.shape))

idx = np.arange(min(length, K+120))
fig, ax = plt.subplots(layout='constrained')
ax.plot(idx/fs, train_x[0,0,idx], color='0.6', alpha=0.6, label='Вход')
ax.plot(idx/fs, train_y[0,0,idx], label='Цель: цифровой фильтр')
ax.set(xlabel='Время, с', ylabel='Значение')
ax.legend()
plt.show()


## 3. Модель
Один Conv1d без bias. Дополнение только слева сохраняет причинность и длину выхода

In [ ]:
class LearnedFIR(nn.Module):
    def __init__(self,taps):
        super().__init__()
        self.taps = taps
        self.conv = nn.Conv1d(1,1,taps,bias=False)

    def forward(self,x):
        return self.conv(F.pad(x,(self.taps-1,0)))

In [ ]:
torch.manual_seed(42)
model = LearnedFIR(K)
with torch.no_grad():
    before = model(val_x)
print(model)
print('Форма выхода:', tuple(before.shape))
print('MSE до обучения:', torch.mean((before[..., K-1:]-val_y[..., K-1:])**2).item())


## 4. Обучение
Веса меняются по train. Лучшее состояние выбирается по validation. Test используется после завершения выбора

Начальные настройки: 200 эпох, learning rate=0.03. При изменении фильтра модель создается заново и ячейки выполняются сверху вниз

In [ ]:
epochs = 200
learning_rate = 0.03

In [ ]:
optimizer = torch.optim.Adam(model.parameters(),lr=learning_rate)
criterion = nn.MSELoss()
train_history, val_history = [],[]
best_loss = float('inf')
best_state = deepcopy(model.state_dict())
for epoch in range(epochs):
    model.train()
    optimizer.zero_grad()
    loss = criterion(model(train_x)[...,K-1:],train_y[...,K-1:])
    loss.backward()
    optimizer.step()
    model.eval()
    with torch.no_grad():
        train_loss = criterion(model(train_x)[...,K-1:],train_y[...,K-1:]).item()
        val_loss = criterion(model(val_x)[...,K-1:],val_y[...,K-1:]).item()
    train_history.append(train_loss)
    val_history.append(val_loss)
    if val_loss < best_loss:
        best_loss = val_loss
        best_state = deepcopy(model.state_dict())
model.load_state_dict(best_state)
print('Лучшая validation MSE:',best_loss)

In [ ]:
import matplotlib.ticker as mticker
fig, ax = plt.subplots(layout='constrained')
ax.semilogy(np.arange(1,len(train_history)+1),train_history,label='Train')
ax.semilogy(np.arange(1,len(val_history)+1),val_history,label='Validation')
ax.yaxis.set_major_formatter(mticker.FuncFormatter(lambda value,pos:f'{value:g}'))
ax.set(xlabel='Эпоха',ylabel='MSE после шага')
ax.legend()
plt.show()

## 5. Проверка на новых записях
Нулевой выход, неизмененный вход и модель сравниваются с одной целью. Первые K-1 отсчетов исключаются у всех вариантов

In [ ]:
test_x,test_y = make_pairs(130,8,length,h)

In [ ]:
model.eval()
with torch.no_grad():
    prediction = model(test_x)
truth = test_y[...,K-1:]
metrics = {
    'Нулевой выход':torch.mean(truth**2).item(),
    'Без фильтрации':torch.mean((test_x[...,K-1:]-truth)**2).item(),
    'Модель':torch.mean((prediction[...,K-1:]-truth)**2).item(),
}
for name,value in metrics.items():
    print(f'{name}: MSE={value:.8g}')

### Вход, выход фильтра и прогноз
Нижний график показывает разность модели и цифрового фильтра

In [ ]:
idx = np.arange(K-1,K+100)
fig, axes = plt.subplots(2,1,figsize=(10,6),sharex=True,layout='constrained')
axes[0].plot(idx/fs,test_x[0,0,idx],color='0.6',alpha=0.65,linewidth=1,label='Вход до фильтрации')
axes[0].plot(idx/fs,test_y[0,0,idx],label='Цифровой фильтр')
axes[0].plot(idx/fs,prediction[0,0,idx],'--',label='Модель')
axes[0].set_ylabel('Значение')
axes[0].legend(fontsize=10,loc='upper right')
axes[1].plot(idx/fs,(prediction-test_y)[0,0,idx])
axes[1].set(xlabel='Время, с',ylabel='Модель - фильтр')
plt.show()

### Спектры на одинаковом участке

In [ ]:
def amplitude(values, fs, hann=False):
    values = np.asarray(values, dtype=float)
    window = np.hanning(len(values)) if hann else np.ones(len(values))
    A = np.abs(np.fft.rfft(values*window))/window.sum()
    if len(values) % 2 == 0:
        A[1:-1] *= 2
    else:
        A[1:] *= 2
    return np.fft.rfftfreq(len(values), d=1/fs), A

In [ ]:
fig, ax = plt.subplots(layout='constrained')
for values,label in [(test_x[0,0,K-1:].numpy(),'Вход до фильтрации'),(test_y[0,0,K-1:].numpy(),'Фильтр'),(prediction[0,0,K-1:].numpy(),'Модель')]:
    f,A = amplitude(values,fs,hann=True)
    ax.plot(f,A,alpha=0.55 if label=='Вход до фильтрации' else 1.0,linestyle='--' if label=='Модель' else '-',label=label)
ax.set(xlabel='Частота, Гц',ylabel='Амплитуда с окном Hann')
ax.legend()
plt.show()

### Коэффициенты и АЧХ

In [ ]:
learned_h = model.conv.weight.detach().numpy()[0,0,::-1].copy()
f,H = freqz(h,fs=fs,worN=2048)
_,H_model = freqz(learned_h,fs=fs,worN=2048)
fig, axes = plt.subplots(1,2,figsize=(11,4),layout='constrained')
axes[0].plot(h,label='Учитель')
axes[0].plot(learned_h,'--',label='Модель')
axes[0].set(xlabel='Номер коэффициента',ylabel='Вес')
axes[0].legend()
axes[1].plot(f,np.abs(H),label='Учитель')
axes[1].plot(f,np.abs(H_model),'--',label='Модель')
axes[1].set(xlabel='Частота, Гц',ylabel='Коэффициент передачи')
axes[1].legend()
plt.show()

## 6. Другая длина записи
Проверяются длина выхода и ошибка на новой записи

In [ ]:
new_x, new_y = make_pairs(140, 1, max(1703,4*K+3), h)
model.eval()
with torch.no_grad():
    new_prediction = model(new_x)
print('Вход:', tuple(new_x.shape), 'Выход:', tuple(new_prediction.shape))
print('MSE:', torch.mean((new_prediction[...,K-1:]-new_y[...,K-1:])**2).item())


## 7. Сохранение и загрузка весов
Файл outputs/learned_filter.pt содержит веса и параметры для повторного запуска

In [ ]:
out = Path('outputs')
out.mkdir(exist_ok=True)
checkpoint = {
    'state_dict': model.state_dict(),
    'taps': K,
    'fs': fs,
    'teacher_h': h.tolist(),
}
torch.save(checkpoint, out/'learned_filter.pt')
saved = torch.load(out/'learned_filter.pt', map_location='cpu', weights_only=True)
restored = LearnedFIR(saved['taps'])
restored.load_state_dict(saved['state_dict'])
restored.eval()
with torch.no_grad():
    matches = torch.allclose(restored(new_x), new_prediction)
print('После загрузки результат совпадает:', matches)


## Выводы

- Выбранный фильтр, полоса и число коэффициентов: ...
- Изменение train и validation MSE при обучении: ...
- Test MSE модели в сравнении с нулевым выходом и неизмененным входом: ...
- Какие частоты ослаблены, насколько совпадают выходы и АЧХ: ...
- Результат проверки другой длины и повторной загрузки: ...
- Ограничения полученного результата: ...
